# Stored execution evidence — RANZCR CLiP

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `82a93db9cfd09fb9ff62a46b1ac1ffb983a1d5ee803b0eb7c4adc453636fe1d6`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab — RANZCR CLiP

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on RANZCR CLiP.

The active flow is unchanged:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the local Kaggle image dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated `run.py` trains the recommended model and saves checkpoints.
6. The submission cell loads the best checkpoint and creates the Kaggle CSV.


## Before running

1. In Colab Secrets add `OPENAI_API_KEY` and `KAGGLE_API_TOKEN`.
2. The Kaggle account owning the token must accept the RANZCR CLiP competition rules.
3. Select a GPU runtime for real training.
4. Run the cells in order.

`EPOCHS = None` is kept in the pipeline cell, so the training cell uses the epoch count recommended by the Agent.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def normalize_repo_url(value: str) -> str:
    value = (value or "").strip()

    markdown_match = re.fullmatch(
        r"\[(.*?)\]\((https?://[^)]+)\)",
        value,
    )
    if markdown_match:
        return markdown_match.group(2)

    plain_url_match = re.search(r"https?://\S+", value)
    if plain_url_match:
        return plain_url_match.group(0)

    return value


REPO_URL = normalize_repo_url(
    os.getenv(
        "JIAOZI_REPO_URL",
        "https://github.com/Isso-W/Jiaozi.git",
    )
)
REPO_REF = os.getenv("JIAOZI_REPO_REF", "main")
REPO_DIR = Path("/content/Jiaozi")

# Important:
# Colab may still be inside /content/Jiaozi from a previous run.
# Move out before deleting and cloning the repository again.
os.chdir("/content")

if REPO_DIR.exists():
    print("Removing existing repository:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    cwd="/content",
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)

print("=== git stderr ===")
print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        "Git clone failed.\n"
        f"Return code: {completed.returncode}\n"
        f"stderr:\n{completed.stderr}"
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

requirements_path = REPO_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        f"requirements.txt was not found at {requirements_path}"
    )

print("\n=== requirements.txt ===")
print(requirements_path.read_text(encoding="utf-8")[:4000])

print("\n=== installing requirements ===")

pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_path),
    ],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)

print("=== pip stdout ===")
print(pip_result.stdout)

print("=== pip stderr ===")
print(pip_result.stderr)

print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError(
        "pip install failed. Check the pip output above."
    )

print("Jiaozi repository:", REPO_DIR)
print("Jiaozi repo URL:", REPO_URL)
print("Jiaozi ref:", REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))

print("pipeline.py candidates:")
for path in pipeline_candidates:
    print(" -", path)

if not pipeline_candidates:
    raise FileNotFoundError(
        "No pipeline.py was found after cloning the repository."
    )


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
=== pip stdout ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 172.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 154.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 147.4 MB/s eta 

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


hf_token = read_secret('HF_TOKEN', required=False)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('Optional HF_TOKEN configured for gated Agent candidates.')
else:
    print('No HF_TOKEN supplied; gated candidates will fall back automatically.')
del hf_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token
Optional HF_TOKEN configured for gated Agent candidates.


## Download the RANZCR CLiP data


In [3]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# Upgrade the packaging tools first
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# Install the Kaggle CLI without -q so failures remain visible
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# Read Kaggle access token from Colab Secrets
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# Verify that the Kaggle CLI is available
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 98.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [4]:
# Configure Kaggle and download RANZCR CLiP dataset
KAGGLE_COMPETITION = "ranzcr-clip-catheter-line-classification"
!kaggle competitions download -c {KAGGLE_COMPETITION} -p /content/data/ranzcr


100% 11.7G/11.7G [01:11<00:00, 176MB/s]



In [5]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data/ranzcr")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip")) or sorted(DATA_ROOT.glob("*.zip"))
print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError("No .zip file found in /content/data/ranzcr")

KAGGLE_DATA_DIR = DATA_ROOT / "extracted"
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

for zip_path in zip_files:
    print("Extracting:", zip_path)
    print("To:", KAGGLE_DATA_DIR)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(KAGGLE_DATA_DIR)

# Keep the original nested-archive handling.
for nested_zip in sorted(KAGGLE_DATA_DIR.rglob("*.zip")):
    nested_target = nested_zip.with_suffix("")
    nested_target.mkdir(parents=True, exist_ok=True)
    print("Extracting nested:", nested_zip, "->", nested_target)
    with zipfile.ZipFile(nested_zip, "r") as archive:
        archive.extractall(nested_target)

required = ["train.csv", "sample_submission.csv", "train", "test"]
missing = [name for name in required if not (KAGGLE_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing extracted RANZCR items: {missing}")

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("Top-level files:")
for path in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", path.name)


Zip files: [PosixPath('/content/data/ranzcr/ranzcr-clip-catheter-line-classification.zip')]
Extracting: /content/data/ranzcr/ranzcr-clip-catheter-line-classification.zip
To: /content/data/ranzcr/extracted
Done.
KAGGLE_DATA_DIR = /content/data/ranzcr/extracted
Top-level files:
 - sample_submission.csv
 - test
 - test_tfrecords
 - train
 - train.csv
 - train_annotations.csv
 - train_tfrecords


## Mount Google Drive for caches and checkpoints


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the unchanged Jiaozi Module 1→4 flow against the local RANZCR files:

1. Convert RANZCR's official multi-label columns into a local training CSV.
2. Run Module 1 from the RANZCR natural-language `QUERY`.
3. Run Module 2 on sampled real chest X-ray images.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with the multi-label training metadata.

`REAL_TRAINING` retains its original meaning. The next cell still trains by executing the generated `run.py`.


In [7]:
# RANZCR Kaggle data -> Jiaozi Module 1-4

import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"
if not PIPELINE_PATH.exists():
    raise FileNotFoundError(f"No pipeline.py found at {PIPELINE_PATH}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

QUERY = (
    "Build a robust multi-label medical image classification pipeline for the "
    "RANZCR CLiP Catheter and Line Position Challenge. Each chest X-ray can have "
    "multiple simultaneous catheter or line labels. Use transfer learning, handle "
    "per-label class imbalance and patient leakage, recommend an appropriate model, "
    "loss, optimizer, augmentation, fine-tuning strategy, batch size and epoch count, "
    "print validation information after every epoch, preserve the best checkpoint, "
    "output independent probabilities for every target, and evaluate with the official "
    "mean ROC AUC across all valid target labels."
)

REAL_TRAINING = True
EPOCHS = None     # Set to 1 for a first test; None uses the Agent recommendation.
OUTPUT_DIR = Path("/content/jiaozi_generated_ranzcr_pipeline")
DATASET_DIR = Path(KAGGLE_DATA_DIR)

raw_train_csv = DATASET_DIR / "train.csv"
sample_submission_csv = DATASET_DIR / "sample_submission.csv"
raw_frame = pd.read_csv(raw_train_csv)
sample_submission = pd.read_csv(sample_submission_csv)

ID_COLUMN = sample_submission.columns[0]
LABEL_COLUMNS = [column for column in sample_submission.columns[1:] if column in raw_frame.columns]
if not LABEL_COLUMNS:
    raise ValueError("Could not identify RANZCR target columns from sample_submission.csv")

if ID_COLUMN not in raw_frame.columns:
    raise ValueError(f"Missing ID column {ID_COLUMN!r} in train.csv")

metadata_columns = [column for column in ["PatientID"] if column in raw_frame.columns]
processed_frame = raw_frame[[ID_COLUMN, *metadata_columns, *LABEL_COLUMNS]].copy()
processed_frame.insert(0, "image", processed_frame[ID_COLUMN].astype(str) + ".jpg")

# Module 2 only needs a categorical field for descriptive profiling.
# Real training below uses every official target column.
processed_frame["analysis_label"] = processed_frame[LABEL_COLUMNS].astype(str).agg("".join, axis=1)
processed_train_csv = DATASET_DIR / "jiaozi_ranzcr_train.csv"
processed_frame.to_csv(processed_train_csv, index=False)

image_dir = DATASET_DIR / "train"
sample_image = image_dir / processed_frame.iloc[0]["image"]
if not sample_image.exists():
    raise FileNotFoundError(sample_image)

LOCAL_DATA_INFO = {
    "benchmark": "ranzcr_clip_catheter_line_classification",
    "competition": "ranzcr-clip-catheter-line-classification",
    "train_csv": str(processed_train_csv),
    "raw_train_csv": str(raw_train_csv),
    "sample_submission_csv": str(sample_submission_csv),
    "image_dir": str(image_dir),
    "test_image_dir": str(DATASET_DIR / "test"),
    "image_column": "image",
    "label_column": "analysis_label",
    "label_columns": LABEL_COLUMNS,
    "id_column": ID_COLUMN,
    "num_classes": len(LABEL_COLUMNS),
    "task_type": "multi_label_image_classification",
    "metric": "mean_roc_auc",
}

print("Raw shape:", raw_frame.shape)
print("Targets:", LABEL_COLUMNS)
print(json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False))

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

CHROMA_DIR = Path("/content/jiaozi_chroma_db")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import derive_recommended_epochs, merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)

print("\n[Notebook] Module 1: Parsing RANZCR requirements...")
m1_output = module1_pipeline(QUERY)
if m1_output is None:
    raise RuntimeError("Module 1 failed.")


class LocalImageSplit:
    column_names = ["image", "label"]

    def __init__(self, frame, info):
        from PIL import Image
        self._Image = Image
        self.labels = frame[info["label_column"]].tolist()
        root = Path(info["image_dir"])
        self.image_paths = [root / str(value) for value in frame[info["image_column"]]]
        missing = [path for path in self.image_paths[:100] if not path.is_file()]
        if missing:
            raise FileNotFoundError(missing[0])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, key):
        if key == "label":
            return self.labels
        if key == "image":
            return [self._open(path) for path in self.image_paths]
        index = int(key)
        return {"image": self._open(self.image_paths[index]), "label": self.labels[index]}

    def _open(self, path):
        with self._Image.open(path) as image:
            return image.convert("RGB")


print("\n[Notebook] Module 2: Analyzing sampled chest X-rays...")
m2_report = ImageStatisticsAnalyzer().analyze(
    {"train": LocalImageSplit(processed_frame, LOCAL_DATA_INFO)}
)

m3_input = merge_modules(m1_output, m2_report)
# Preserve Module 1's inferred task type; only attach dataset facts.
m3_input["multi_label"] = True
m3_input["evaluation_metric"] = "mean_roc_auc"
m3_input["num_classes"] = len(LABEL_COLUMNS)
m3_input["label_columns"] = LABEL_COLUMNS

print("\n[Notebook] Module 3: Retrieving model configurations...")
graph = build_graph()
collection = build_vector_index(persist_path=str(CHROMA_DIR))
recommendations = retrieve_top3_hybrid(m3_input, graph, collection)
print_results(m3_input, recommendations, graph)

try:
    recommendations = recommend(
        recommendations, m2_report, m3_input, memory=OutcomeMemory()
    )
    print("[Notebook] Recommender re-ranked candidates.")
except Exception as exc:
    print(f"[Notebook] Recommender skipped: {exc}")

task_lists = build_all_task_lists(recommendations, graph, fmt="nl")
if not task_lists:
    raise RuntimeError("Module 3 returned no task lists.")

for task_list in task_lists:
    model_config = task_list.get("model_config")
    if not isinstance(model_config, dict):
        continue
    model_config.update(
        {
            "num_classes": len(LABEL_COLUMNS),
            "train_csv": str(processed_train_csv),
            "image_dir": str(image_dir),
            "image_column": "image",
            "label_columns": LABEL_COLUMNS,
            "id_column": ID_COLUMN,
            "multi_label": True,
            "evaluation_metric": "mean_roc_auc",
            "metric": "mean_roc_auc",
            "metric_name": "mean_roc_auc",
            "offline_smoke": not REAL_TRAINING,
            "benchmark_key": LOCAL_DATA_INFO["benchmark"],
            "competition": LOCAL_DATA_INFO["competition"],
        }
    )
    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get("data_size", "large"),
            model_config.get("finetune_strategy"),
            bool(model_config.get("pretrained_hf_id")),
        ),
    )

print("\nTop model config preview:")
print(json.dumps(task_lists[0]["model_config"], indent=2, ensure_ascii=False)[:4000])

print("\n[Notebook] Module 4: Generating project...")
module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="openai",
)

summary_path = OUTPUT_DIR / "module4_summary.json"
if not summary_path.exists():
    summary_path.write_text(
        json.dumps(module4["summary"], indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

# Store the immutable task contract next to the generated project.
(OUTPUT_DIR / "ranzcr_contract.json").write_text(
    json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Module 4 summary:", summary_path)
print("Generated project:", OUTPUT_DIR)

# -------------------------------------------------
# 15. RANZCR multi-label compatibility adapter
# -------------------------------------------------
# Preserve the Agent-generated project, then add a task-contract adapter.
# Backbone, optimizer, finetune strategy and epochs remain Agent-selected.
adapter_path = OUTPUT_DIR / "ranzcr_multilabel_adapter.py"
adapter_path.write_text('# Runtime adapter for RANZCR multi-label training.\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupShuffleSplit, train_test_split\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\nfrom transformers import AutoConfig, AutoImageProcessor, AutoModel\n\n\nROOT = Path(__file__).resolve().parent\nCONFIG_PATH = ROOT / "configs.json"\nCONTRACT_PATH = ROOT / "ranzcr_contract.json"\n\n\ndef load_inputs():\n    configs = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))\n    contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))\n    if not configs:\n        raise ValueError("configs.json is empty")\n    return configs, contract\n\n\ndef flatten(config):\n    return {**config, **(config.get("model_config") or {})}\n\n\ndef candidate_hf_id(config):\n    value = flatten(config)\n    return value.get("pretrained_hf_id") or value.get("model_name")\n\n\ndef select_loadable_candidate(configs):\n    failures = []\n    for index, config in enumerate(configs):\n        flat = flatten(config)\n        hf_id = candidate_hf_id(config)\n        if not hf_id:\n            failures.append(f"candidate {index + 1}: missing pretrained_hf_id")\n            continue\n        try:\n            processor = AutoImageProcessor.from_pretrained(hf_id)\n            backbone = AutoModel.from_pretrained(hf_id)\n            print(f"[adapter] Selected Agent candidate {index + 1}: {hf_id}", flush=True)\n            return index, config, flat, processor, backbone\n        except Exception as exc:\n            failures.append(f"candidate {index + 1} ({hf_id}): {type(exc).__name__}: {exc}")\n            print(f"[adapter] Candidate unavailable, trying next: {hf_id}", flush=True)\n    raise RuntimeError("No Agent candidate could be loaded:\\n" + "\\n".join(failures))\n\n\ndef image_size_from_processor(processor, config):\n    configured = flatten(config).get("image_size")\n    if configured:\n        return int(configured)\n    size = getattr(processor, "size", {}) or {}\n    if isinstance(size, int):\n        return size\n    return int(size.get("height") or size.get("shortest_edge") or 224)\n\n\ndef processor_stats(processor):\n    mean = getattr(processor, "image_mean", None) or [0.485, 0.456, 0.406]\n    std = getattr(processor, "image_std", None) or [0.229, 0.224, 0.225]\n    return mean, std\n\n\ndef hidden_size(backbone):\n    config = backbone.config\n    for name in ("hidden_size", "embed_dim", "hidden_sizes"):\n        value = getattr(config, name, None)\n        if isinstance(value, int):\n            return value\n        if isinstance(value, (list, tuple)) and value:\n            return int(value[-1])\n    raise ValueError(f"Could not determine hidden size from {type(config).__name__}")\n\n\ndef pooled_features(outputs):\n    pooled = getattr(outputs, "pooler_output", None)\n    if pooled is not None:\n        return pooled\n    last = getattr(outputs, "last_hidden_state", None)\n    if last is None and isinstance(outputs, (tuple, list)):\n        last = outputs[0]\n    if last is None:\n        raise ValueError("Backbone output has no pooler_output or last_hidden_state")\n    if last.ndim == 4:\n        return last.mean(dim=(-2, -1))\n    if last.ndim == 3:\n        return last[:, 0]\n    return last\n\n\nclass MultiLabelModel(nn.Module):\n    def __init__(self, backbone, num_labels, dropout=0.1):\n        super().__init__()\n        self.backbone = backbone\n        self.classifier = nn.Sequential(\n            nn.Dropout(dropout),\n            nn.Linear(hidden_size(backbone), num_labels),\n        )\n\n    def forward(self, pixel_values):\n        outputs = self.backbone(pixel_values=pixel_values)\n        return self.classifier(pooled_features(outputs))\n\n\ndef apply_finetune_strategy(model, config):\n    flat = flatten(config)\n    strategy = str(flat.get("finetune_strategy") or "full").lower()\n    if strategy == "full":\n        print("[adapter] Finetune strategy: full", flush=True)\n        return\n\n    for parameter in model.backbone.parameters():\n        parameter.requires_grad = False\n\n    if strategy == "partial":\n        n_blocks = int(flat.get("unfreeze_last_n_blocks") or 4)\n        blocks = None\n        for path in (\n            ("encoder", "layer"),\n            ("encoder", "layers"),\n            ("layer",),\n            ("layers",),\n        ):\n            value = model.backbone\n            for name in path:\n                value = getattr(value, name, None)\n                if value is None:\n                    break\n            if value is not None and hasattr(value, "__len__"):\n                blocks = value\n                break\n        if blocks is not None:\n            for block in list(blocks)[-n_blocks:]:\n                for parameter in block.parameters():\n                    parameter.requires_grad = True\n        else:\n            named = list(model.backbone.named_parameters())\n            start = int(len(named) * 0.75)\n            for _, parameter in named[start:]:\n                parameter.requires_grad = True\n\n    if flat.get("train_norm_layers", True):\n        for name, parameter in model.backbone.named_parameters():\n            if "norm" in name.lower():\n                parameter.requires_grad = True\n\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    print(f"[adapter] Finetune strategy: {strategy}; trainable={trainable:,}/{total:,}", flush=True)\n\n\nclass RanzcrDataset(Dataset):\n    def __init__(self, frame, image_dir, id_column, label_columns, transform, with_labels=True):\n        self.frame = frame.reset_index(drop=True)\n        self.image_dir = Path(image_dir)\n        self.id_column = id_column\n        self.label_columns = label_columns\n        self.transform = transform\n        self.with_labels = with_labels\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, index):\n        row = self.frame.iloc[index]\n        image_id = str(row[self.id_column])\n        with Image.open(self.image_dir / f"{image_id}.jpg") as image:\n            pixel_values = self.transform(image.convert("RGB"))\n        if not self.with_labels:\n            return pixel_values, image_id\n        target = torch.tensor(row[self.label_columns].to_numpy(dtype=np.float32))\n        return pixel_values, target\n\n\nclass BinaryFocalLoss(nn.Module):\n    def __init__(self, gamma=2.0, pos_weight=None):\n        super().__init__()\n        self.gamma = gamma\n        self.register_buffer("pos_weight", pos_weight)\n\n    def forward(self, logits, targets):\n        base = nn.functional.binary_cross_entropy_with_logits(\n            logits, targets, reduction="none", pos_weight=self.pos_weight\n        )\n        probability = torch.sigmoid(logits)\n        pt = probability * targets + (1 - probability) * (1 - targets)\n        return (((1 - pt) ** self.gamma) * base).mean()\n\n\ndef build_loss(config, targets, device):\n    flat = flatten(config)\n    requested = str(flat.get("loss") or flat.get("loss_name") or "").lower()\n    positive = targets.sum(axis=0).astype(np.float32)\n    negative = len(targets) - positive\n    weights = np.clip(negative / np.maximum(positive, 1.0), 1.0, 20.0)\n    pos_weight = torch.tensor(weights, device=device)\n    if "focal" in requested:\n        print("[adapter] Compatible loss: multi-label binary focal loss", flush=True)\n        return BinaryFocalLoss(gamma=2.0, pos_weight=pos_weight)\n    if "cross_entropy" in requested:\n        print(\n            "[adapter] Rejected incompatible single-label CrossEntropyLoss; "\n            "using weighted BCEWithLogitsLoss while preserving all other Agent choices.",\n            flush=True,\n        )\n    else:\n        print("[adapter] Compatible loss: weighted BCEWithLogitsLoss", flush=True)\n    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n\n\ndef build_transforms(processor, image_size):\n    mean, std = processor_stats(processor)\n    train_transform = transforms.Compose([\n        transforms.Resize((image_size, image_size)),\n        transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(5),\n        transforms.ColorJitter(brightness=0.1, contrast=0.1),\n        transforms.ToTensor(),\n        transforms.Normalize(mean, std),\n    ])\n    eval_transform = transforms.Compose([\n        transforms.Resize((image_size, image_size)),\n        transforms.ToTensor(),\n        transforms.Normalize(mean, std),\n    ])\n    return train_transform, eval_transform\n\n\ndef split_frame(frame, patient_column, seed=42):\n    if patient_column and patient_column in frame.columns:\n        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)\n        train_index, val_index = next(splitter.split(frame, groups=frame[patient_column]))\n        print(f"[adapter] Patient-group split using {patient_column}", flush=True)\n    else:\n        train_index, val_index = train_test_split(\n            np.arange(len(frame)), test_size=0.2, random_state=seed\n        )\n        print("[adapter] WARNING: PatientID unavailable; using image-level split", flush=True)\n    return frame.iloc[train_index].copy(), frame.iloc[val_index].copy()\n\n\ndef calculate_auc(y_true, y_probability, label_columns):\n    scores = {}\n    for index, label in enumerate(label_columns):\n        truth = y_true[:, index]\n        if np.unique(truth).size < 2:\n            continue\n        scores[label] = float(roc_auc_score(truth, y_probability[:, index]))\n    if not scores:\n        raise RuntimeError("No validation label has both classes")\n    return float(np.mean(list(scores.values()))), scores\n\n\ndef checkpoint_dict(model, optimizer, scheduler, epoch, best_auc, validation, selected, image_size, labels, id_column):\n    return {\n        "epoch": epoch,\n        "best_epoch": epoch,\n        "best_metric": best_auc,\n        "metric_name": "mean_roc_auc",\n        "validation": validation,\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scheduler_state_dict": scheduler.state_dict(),\n        "selected_candidate_index": selected,\n        "image_size": image_size,\n        "label_columns": labels,\n        "id_column": id_column,\n    }\n\n\ndef prepare(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    configs, contract = load_inputs()\n    selected, config, flat, processor, backbone = select_loadable_candidate(configs)\n    labels = contract["label_columns"]\n    id_column = contract["id_column"]\n    frame = pd.read_csv(contract["raw_train_csv"])\n    patient_column = "PatientID" if "PatientID" in frame.columns else None\n    train_frame, val_frame = split_frame(frame, patient_column, seed)\n    image_size = image_size_from_processor(processor, config)\n    train_transform, eval_transform = build_transforms(processor, image_size)\n    return configs, contract, selected, config, flat, backbone, train_frame, val_frame, image_size, train_transform, eval_transform\n\n\ndef smoke_test():\n    (_, contract, selected, config, flat, backbone, train_frame, _, image_size, train_transform, _) = prepare()\n    labels = contract["label_columns"]\n    dataset = RanzcrDataset(\n        train_frame.head(4), contract["image_dir"], contract["id_column"], labels, train_transform\n    )\n    images, targets = next(iter(DataLoader(dataset, batch_size=2)))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = MultiLabelModel(backbone, len(labels)).to(device)\n    apply_finetune_strategy(model, config)\n    criterion = build_loss(config, train_frame[labels].to_numpy(), device)\n    logits = model(images.to(device))\n    loss = criterion(logits, targets.to(device))\n    loss.backward()\n    probabilities = torch.sigmoid(logits)\n    assert logits.shape == targets.shape == (2, len(labels))\n    assert torch.isfinite(loss)\n    assert bool(((probabilities >= 0) & (probabilities <= 1)).all())\n    print(\n        f"[smoke] success images={tuple(images.shape)} targets={tuple(targets.shape)} "\n        f"logits={tuple(logits.shape)} loss={loss.item():.6f} candidate={selected + 1}",\n        flush=True,\n    )\n\n\ndef train(epochs):\n    if not torch.cuda.is_available():\n        raise RuntimeError("GPU runtime required for real training")\n    torch.backends.cudnn.benchmark = True\n    (configs, contract, selected, config, flat, backbone, train_frame, val_frame, image_size, train_transform, eval_transform) = prepare()\n    labels = contract["label_columns"]\n    id_column = contract["id_column"]\n    batch_size = int(flat.get("batch_size") or 16)\n    learning_rate = float(flat.get("learning_rate") or 1e-4)\n    weight_decay = float(flat.get("weight_decay") or 1e-4)\n    workers = min(4, os.cpu_count() or 2)\n    train_loader = DataLoader(\n        RanzcrDataset(train_frame, contract["image_dir"], id_column, labels, train_transform),\n        batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True,\n        persistent_workers=workers > 0,\n    )\n    val_loader = DataLoader(\n        RanzcrDataset(val_frame, contract["image_dir"], id_column, labels, eval_transform),\n        batch_size=batch_size * 2, shuffle=False, num_workers=workers, pin_memory=True,\n        persistent_workers=workers > 0,\n    )\n    device = torch.device("cuda")\n    model = MultiLabelModel(backbone, len(labels)).to(device)\n    apply_finetune_strategy(model, config)\n    criterion = build_loss(config, train_frame[labels].to_numpy(), device)\n    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]\n    optimizer_name = str(flat.get("optimizer") or "adamw").lower()\n    if optimizer_name == "sgd":\n        optimizer = torch.optim.SGD(trainable, lr=learning_rate, momentum=0.9, weight_decay=weight_decay)\n    else:\n        optimizer = torch.optim.AdamW(trainable, lr=learning_rate, weight_decay=weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))\n    scaler = torch.amp.GradScaler("cuda")\n    checkpoint_dir = Path(flat.get("checkpoint_dir") or ROOT / "checkpoints")\n    important_dir = checkpoint_dir / "important_epochs"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    important_dir.mkdir(parents=True, exist_ok=True)\n    history = []\n    best_auc = -math.inf\n    best_epoch = 0\n    print(\n        f"[train] candidate={selected + 1} backbone={candidate_hf_id(config)} epochs={epochs} "\n        f"batch={batch_size} image_size={image_size} optimizer={optimizer_name}", flush=True\n    )\n    for epoch in range(1, epochs + 1):\n        started = time.time()\n        model.train()\n        total_loss = 0.0\n        steps = 0\n        for images, targets in train_loader:\n            images = images.to(device, non_blocking=True)\n            targets = targets.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.amp.autocast("cuda"):\n                logits = model(images)\n                loss = criterion(logits, targets)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            total_loss += float(loss.item())\n            steps += 1\n        model.eval()\n        val_loss = 0.0\n        val_steps = 0\n        truths, probabilities = [], []\n        with torch.inference_mode():\n            for images, targets in val_loader:\n                images = images.to(device, non_blocking=True)\n                gpu_targets = targets.to(device, non_blocking=True)\n                with torch.amp.autocast("cuda"):\n                    logits = model(images)\n                    loss = criterion(logits, gpu_targets)\n                val_loss += float(loss.item())\n                val_steps += 1\n                truths.append(targets.numpy())\n                probabilities.append(torch.sigmoid(logits).float().cpu().numpy())\n        y_true = np.concatenate(truths)\n        y_probability = np.concatenate(probabilities)\n        mean_auc, per_label = calculate_auc(y_true, y_probability, labels)\n        train_loss = total_loss / max(steps, 1)\n        validation_loss = val_loss / max(val_steps, 1)\n        lr = optimizer.param_groups[0]["lr"]\n        elapsed = time.time() - started\n        record = {\n            "epoch": epoch, "train_loss": train_loss, "val_loss": validation_loss,\n            "val_mean_auc": mean_auc, "lr": lr, "steps": steps, "time_seconds": elapsed,\n        }\n        history.append(record)\n        pd.DataFrame(history).to_csv(checkpoint_dir / "training_history.csv", index=False)\n        print(\n            f"[train] epoch {epoch}/{epochs} loss={train_loss:.6f} val_loss={validation_loss:.6f} "\n            f"val_mean_auc={mean_auc:.6f} lr={lr:.2e} steps={steps} time={elapsed:.1f}s",\n            flush=True,\n        )\n        validation = {\n            "mean_roc_auc": mean_auc, "val_loss": validation_loss,\n            "per_label_auc": per_label, "num_samples": len(val_frame),\n        }\n        payload = checkpoint_dict(\n            model, optimizer, scheduler, epoch, max(best_auc, mean_auc), validation,\n            selected, image_size, labels, id_column,\n        )\n        torch.save(payload, checkpoint_dir / "latest_model.pt")\n        if mean_auc > best_auc:\n            best_auc = mean_auc\n            best_epoch = epoch\n            payload["best_metric"] = best_auc\n            payload["best_epoch"] = best_epoch\n            best_path = checkpoint_dir / "best_model.pt"\n            torch.save(payload, best_path)\n            representative = important_dir / f"best_epoch_{epoch:03d}.pt"\n            shutil.copy2(best_path, representative)\n            saved_representatives = sorted(\n                important_dir.glob("best_epoch_*.pt"),\n                key=lambda path: path.stat().st_mtime,\n                reverse=True,\n            )\n            for stale_path in saved_representatives[3:]:\n                stale_path.unlink()\n                print(\n                    f"[pipeline] Removed older representative to protect Drive quota: {stale_path}",\n                    flush=True,\n                )\n            print(f"[train] Saved new best checkpoint at epoch {epoch}: {best_path}", flush=True)\n            print(f"[pipeline] Preserved representative checkpoint: {representative}", flush=True)\n        scheduler.step()\n    print(f"[train] complete best_epoch={best_epoch} best_mean_auc={best_auc:.6f}", flush=True)\n\n\ndef predict(output_path):\n    configs, contract = load_inputs()\n    flat_first = flatten(configs[0])\n    checkpoint_dir = Path(flat_first.get("checkpoint_dir") or ROOT / "checkpoints")\n    checkpoint_path = checkpoint_dir / "best_model.pt"\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    selected = int(checkpoint["selected_candidate_index"])\n    config = configs[selected]\n    hf_id = candidate_hf_id(config)\n    processor = AutoImageProcessor.from_pretrained(hf_id)\n    backbone = AutoModel.from_pretrained(hf_id)\n    labels = checkpoint["label_columns"]\n    id_column = checkpoint["id_column"]\n    _, eval_transform = build_transforms(processor, int(checkpoint["image_size"]))\n    model = MultiLabelModel(backbone, len(labels))\n    model.load_state_dict(checkpoint["model_state_dict"])\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = model.to(device).eval()\n    sample = pd.read_csv(contract["sample_submission_csv"])\n    loader = DataLoader(\n        RanzcrDataset(sample, contract["test_image_dir"], id_column, labels, eval_transform, False),\n        batch_size=32, shuffle=False, num_workers=min(4, os.cpu_count() or 2),\n        pin_memory=torch.cuda.is_available(),\n    )\n    ids, values = [], []\n    with torch.inference_mode():\n        for images, batch_ids in loader:\n            logits = model(images.to(device, non_blocking=True))\n            values.append(torch.sigmoid(logits).cpu().numpy())\n            ids.extend(batch_ids)\n    result = pd.DataFrame(np.concatenate(values), columns=labels)\n    result.insert(0, id_column, ids)\n    submission = sample[[id_column]].merge(result, on=id_column, how="left")\n    assert submission.shape == sample.shape\n    assert list(submission.columns) == list(sample.columns)\n    assert submission[labels].notna().all().all()\n    assert ((submission[labels] >= 0) & (submission[labels] <= 1)).all().all()\n    Path(output_path).parent.mkdir(parents=True, exist_ok=True)\n    submission.to_csv(output_path, index=False)\n    print(f"[predict] wrote {output_path} shape={submission.shape}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--epochs", type=int)\n    parser.add_argument("--smoke-test", action="store_true")\n    parser.add_argument("--predict")\n    args = parser.parse_args()\n    if args.smoke_test:\n        smoke_test()\n    elif args.predict:\n        predict(args.predict)\n    elif args.epochs:\n        train(args.epochs)\n    else:\n        parser.error("choose --smoke-test, --epochs, or --predict")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print("RANZCR compatibility adapter:", adapter_path)

# Real mini-batch contract test: forward, compatible loss, backward, sigmoid.
smoke_command = [sys.executable, "-u", str(adapter_path), "--smoke-test"]
print("Running real RANZCR contract smoke test...")
smoke_result = subprocess.run(
    smoke_command, cwd=str(OUTPUT_DIR), text=True, capture_output=True
)
print(smoke_result.stdout)
print(smoke_result.stderr)
if smoke_result.returncode != 0:
    raise RuntimeError(
        "RANZCR compatibility smoke test failed. Do not start full training."
    )


Raw shape: (30083, 13)
Targets: ['ETT - Abnormal', 'ETT - Borderline', 'ETT - Normal', 'NGT - Abnormal', 'NGT - Borderline', 'NGT - Incompletely Imaged', 'NGT - Normal', 'CVC - Abnormal', 'CVC - Borderline', 'CVC - Normal', 'Swan Ganz Catheter Present']
{
  "benchmark": "ranzcr_clip_catheter_line_classification",
  "competition": "ranzcr-clip-catheter-line-classification",
  "train_csv": "/content/data/ranzcr/extracted/jiaozi_ranzcr_train.csv",
  "raw_train_csv": "/content/data/ranzcr/extracted/train.csv",
  "sample_submission_csv": "/content/data/ranzcr/extracted/sample_submission.csv",
  "image_dir": "/content/data/ranzcr/extracted/train",
  "test_image_dir": "/content/data/ranzcr/extracted/test",
  "image_column": "image",
  "label_column": "analysis_label",
  "label_columns": [
    "ETT - Abnormal",
    "ETT - Borderline",
    "ETT - Normal",
    "NGT - Abnormal",
    "NGT - Borderline",
    "NGT - Incompletely Imaged",
    "NGT - Normal",
    "CVC - Abnormal",
    "CVC - Borderlin

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : classification
Input  : size=medium  priority=accuracy
Flags  : class_imbalance, medical
Desc   : Build a robust multi-label medical image classification pipeline for the RANZCR CLiP Catheter and Line Position Challenge. Each chest X-ray can have multiple simultaneous catheter or line labels. Use transfer learning, handle per-label class imbalance and patient leakage, recommend an appropriate model, loss, optimizer, augmentation, fine-tuning strategy, batch size and epoch count, print validation information after every epoch, preserve the best checkpoint, output independent probabilities for every target, and evaluate with the official mean ROC AUC across all valid target labels.
──────────────────────────────────────────────────────────────────────

Top 1  [score=0.898  struct=1.0  vec=0.745]
  backbone  : DINOv3
  head      : Classificatio

In [8]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 0.898,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 0.898,
      "task_type": "classification"
    },
    {
      "backbone": "dinov2",
      "finetune_strategy": "full",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.779,
      "task_type": "classification"
    },
    {
      "backbone": "swin_transformer",
      "finetune_strategy": "full",
      "loss": "focal_loss",
      "optimizer": "adamw",
      "rank": 4,
      "score": 0.743,
      "task_type": "classification"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evaluate.py",
   

## Train the recommended model (real data)

This cell keeps the original training responsibility. It reads the Agent-recommended epoch count and configuration, then starts the RANZCR compatibility runner created after Module 4. The adapter preserves the selected backbone, optimizer and fine-tuning strategy, but rejects mathematically incompatible single-label loss/output behavior. Every epoch is streamed and the best checkpoint is preserved.


In [9]:
# Train the Agent-recommended RANZCR candidate through the verified adapter.
import json
import os
import subprocess
import sys
from pathlib import Path

if not REAL_TRAINING:
    print("REAL_TRAINING is False - skipping the real training step.")
else:
    cfg_path = OUTPUT_DIR / "configs.json"
    configs = json.loads(cfg_path.read_text(encoding="utf-8"))
    if not configs:
        raise ValueError("configs.json is empty.")

    cfg = configs[0]
    mc = cfg.get("model_config", cfg)
    epochs = EPOCHS
    if epochs is None:
        epochs = mc.get("recommended_epochs") or cfg.get("recommended_epochs") or 10

    ckpt_dir = "/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2"
    if not Path("/content/drive/MyDrive").exists():
        raise RuntimeError("Google Drive is not mounted.")
    os.makedirs(ckpt_dir, exist_ok=True)

    # Only execution/checkpoint metadata is added; Agent recommendations remain.
    for item in configs:
        item["checkpoint_dir"] = ckpt_dir
        item["resume_checkpoint"] = None
        if isinstance(item.get("model_config"), dict):
            item["model_config"]["checkpoint_dir"] = ckpt_dir
            item["model_config"]["resume_checkpoint"] = None
    cfg_path.write_text(json.dumps(configs, indent=2, ensure_ascii=False), encoding="utf-8")

    print("Agent-recommended backbone:", mc.get("backbone"))
    print("Agent-recommended loss:", mc.get("loss"))
    print("Agent-recommended optimizer:", mc.get("optimizer"))
    print("Agent-recommended finetune strategy:", mc.get("finetune_strategy"))
    print("Agent-recommended epochs:", epochs)
    print("Checkpoint directory:", ckpt_dir)

    train_command = [
        sys.executable, "-u", "ranzcr_multilabel_adapter.py",
        "--epochs", str(epochs),
    ]
    print("Command:", " ".join(train_command), "(cwd:", OUTPUT_DIR, ")")
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        train_command, cwd=str(OUTPUT_DIR), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
    )
    if process.stdout is None:
        raise RuntimeError("Could not capture training output.")
    for line in iter(process.stdout.readline, ""):
        print(line, end="", flush=True)
    process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, train_command)
    print("\nReal training finished.")
    print("Checkpoints under:", ckpt_dir)


Agent-recommended backbone: dinov3
Agent-recommended loss: cross_entropy_loss
Agent-recommended optimizer: adamw
Agent-recommended finetune strategy: partial
Agent-recommended epochs: 20
Checkpoint directory: /content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2
Command: /usr/bin/python3 -u ranzcr_multilabel_adapter.py --epochs 20 (cwd: /content/jiaozi_generated_ranzcr_pipeline )

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 10609.71it/s]
[adapter] Selected Agent candidate 1: facebook/dinov3-vitb16-pretrain-lvd1689m
[adapter] Patient-group split using PatientID
[adapter] Finetune strategy: partial; trainable=21,303,563/85,668,875
[adapter] Rejected incompatible single-label CrossEntropyLoss; using weighted BCEWithLogitsLoss while preserving all other Agent choices.
[train] candidate=1 backbone=facebook/dinov3-vitb16-pretrain-lvd1689m epochs=20 batch=16 image_size=224 optimizer=adamw
[train] epoch 1/20 loss=0.685636 val_loss=0.628956 val_mean_auc=0.793603 lr=1.0

In [ ]:
!nvidia-smi


Sat Jul 11 18:36:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             33W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Show the result

Reads the best checkpoint and prints its saved RANZCR validation mean ROC AUC instantly. No additional training is performed in this cell.


In [10]:
import json
import torch
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2")
best_path = ckpt_dir / "best_model.pt"
if not best_path.exists():
    raise FileNotFoundError(best_path)

checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
print("=== RESULT (best checkpoint) ===")
print("checkpoint :", best_path)
print("best_epoch :", checkpoint.get("best_epoch", checkpoint.get("epoch")))
print("best_metric:", checkpoint.get("best_metric"), "(mean ROC AUC; higher is better)")
print("validation :", json.dumps(checkpoint.get("validation"), indent=2, ensure_ascii=False))

import pandas as pd
history_path = ckpt_dir / "training_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)
    history.plot(x="epoch", y=["train_loss", "val_loss"], grid=True, figsize=(10, 4))
    history.plot(x="epoch", y="val_mean_auc", grid=True, figsize=(10, 4))


=== RESULT (best checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/best_model.pt
best_epoch : 16
best_metric: 0.8577258824877178 (mean ROC AUC; higher is better)
validation : {
  "mean_roc_auc": 0.8577258824877178,
  "val_loss": 0.4616491648283872,
  "per_label_auc": {
    "ETT - Abnormal": 0.944895038167939,
    "ETT - Borderline": 0.9346101424361493,
    "ETT - Normal": 0.9846893382286153,
    "NGT - Abnormal": 0.9073529815694783,
    "NGT - Borderline": 0.9032883002313923,
    "NGT - Incompletely Imaged": 0.9649396764789759,
    "NGT - Normal": 0.9563931943630619,
    "CVC - Abnormal": 0.6351496438735087,
    "CVC - Borderline": 0.6149905651369305,
    "CVC - Normal": 0.5972551161803242,
    "Swan Ganz Catheter Present": 0.9914207106985208
  },
  "num_samples": 5250
}


,epoch,train_loss,val_loss,val_mean_auc,lr,steps,time_seconds
0,1,0.685636,0.628956,0.793603,0.001000,1553,173.362030
1,2,0.623066,0.549282,0.807336,0.000994,1553,176.111005
2,3,0.596110,0.548684,0.815726,0.000976,1553,180.146919
3,4,0.575378,0.526446,0.826269,0.000946,1553,174.683068
4,5,0.556708,0.510722,0.836383,0.000905,1553,172.267269
5,6,0.548534,0.530743,0.833915,0.000854,1553,170.911507
6,7,0.539660,0.495821,0.843685,0.000794,1553,169.562373
7,8,0.526719,0.513020,0.841576,0.000727,1553,172.165806
8,9,0.516555,0.503853,0.843798,0.000655,1553,171.367363
9,10,0.509703,0.477033,0.849431,0.000578,1553,175.417710


<Figure size 1000x400 with 1 Axes>

<Figure size 1000x400 with 1 Axes>

## Package the trained model for a Kaggle Notebook submission

RANZCR CLiP is a **code/Notebook submission** competition. A raw CSV sent from Colab is rejected with HTTP 400. This section follows the same two-file workflow used for Aerial Cactus: it packages the existing best checkpoint as a private Kaggle Dataset asset, while the separate `RANZCR_CLiP_Jiaozi_Kaggle_Final.ipynb` performs test inference inside Kaggle and writes `/kaggle/working/submission.csv`.

After a Colab disconnect, only run the two cells below. They read the existing checkpoint from Google Drive and do not retrain or rerun Modules 1–4.

### Retraining storage safety

This integrated version uses a fresh Drive directory `ranzcr_agent_retrained_v2`. The model, Agent-selected epochs, loss, optimizer, fine-tuning strategy and validation metric are unchanged. Only redundant representative checkpoint copies are capped at the latest three improvements; `best_model.pt` and `latest_model.pt` are always retained. Allow at least 4 GB of free Drive space before training.


In [11]:
# Create the Kaggle Dataset asset zip from the saved Drive checkpoint.
# No training and no Agent/API call occur in this cell.

import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

try:
    from google.colab import drive, userdata
except Exception as exc:
    raise RuntimeError("Run this packaging cell in Google Colab.") from exc

drive.mount("/content/drive")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers"],
    check=True,
)

HF_CACHE_DIR = Path("/content/drive/MyDrive/Jiaozi/hf_cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

from transformers import AutoConfig

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/Jiaozi/checkpoints/"
    "ranzcr_agent_retrained_v2"
)
ASSET_DIR = Path("/content/ranzcr_dinov3_assets")
ASSET_ZIP = Path("/content/ranzcr_dinov3_assets.zip")

if not CHECKPOINT_DIR.is_dir():
    raise FileNotFoundError(
        f"Saved checkpoint directory not found: {CHECKPOINT_DIR}. "
        "Mount the same Drive account used for training."
    )

candidate_paths = []
for path in [
    CHECKPOINT_DIR / "best_model.pt",
    CHECKPOINT_DIR / "latest_model.pt",
    *sorted((CHECKPOINT_DIR / "important_epochs").glob("*.pt")),
]:
    if path.is_file() and path not in candidate_paths:
        candidate_paths.append(path)
if not candidate_paths:
    raise FileNotFoundError(f"No checkpoint files found under {CHECKPOINT_DIR}")

scored = []
for path in candidate_paths:
    candidate = torch.load(path, map_location="cpu", weights_only=False)
    validation = candidate.get("validation") or {}
    current_auc = validation.get("mean_roc_auc")
    if current_auc is None:
        current_auc = validation.get("metric_value")
    if current_auc is not None:
        scored.append(
            (
                float(current_auc),
                int(candidate.get("epoch", 0)),
                path,
            )
        )
    del candidate
    gc.collect()
if not scored:
    raise RuntimeError("No checkpoint contains its own validation ROC-AUC")
scored.sort(key=lambda item: item[0], reverse=True)
print("Checkpoint candidates ranked by their own validation mean ROC-AUC:")
for auc, epoch, path in scored:
    print(f" - epoch={epoch} auc={auc:.12f} path={path}")
_, _, CHECKPOINT_PATH = scored[0]

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
selected_validation = checkpoint.get("validation") or {}
selected_auc = selected_validation.get("mean_roc_auc")
if selected_auc is None:
    selected_auc = selected_validation.get("metric_value")
print("Selected checkpoint:", CHECKPOINT_PATH)
print("Selected epoch:", checkpoint.get("epoch"))
print("Selected validation mean ROC-AUC:", selected_auc)
required = {"model_state_dict", "image_size", "label_columns", "id_column"}
missing = sorted(required.difference(checkpoint))
if missing:
    raise KeyError(f"Checkpoint is missing required fields: {missing}")

label_columns = list(checkpoint["label_columns"])
if len(label_columns) != 11:
    raise ValueError(f"Expected 11 RANZCR labels, found {len(label_columns)}")

if ASSET_DIR.exists():
    shutil.rmtree(ASSET_DIR)
if ASSET_ZIP.exists():
    ASSET_ZIP.unlink()
ASSET_DIR.mkdir(parents=True, exist_ok=True)

print("Copying checkpoint to local Colab storage...")
shutil.copy2(CHECKPOINT_PATH, ASSET_DIR / "best_model.pt")

hf_token = userdata.get("HF_TOKEN") or None
config = AutoConfig.from_pretrained(MODEL_ID, token=hf_token)
config.save_pretrained(ASSET_DIR)

manifest = {
    "competition": "ranzcr-clip-catheter-line-classification",
    "model_id": MODEL_ID,
    "checkpoint_file": "best_model.pt",
    "image_size": int(checkpoint["image_size"]),
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
    "id_column": str(checkpoint["id_column"]),
    "label_columns": label_columns,
    "best_epoch": int(checkpoint.get("best_epoch", checkpoint.get("epoch", 0))),
    "best_metric": float(selected_auc),
    "metric_name": "mean_roc_auc",
    "inference": "sigmoid probabilities for 11 independent labels",
}

(ASSET_DIR / "asset_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Creating asset zip...")
shutil.make_archive(
    str(ASSET_ZIP.with_suffix("")),
    "zip",
    root_dir=ASSET_DIR,
)

sha256 = hashlib.sha256()
with ASSET_ZIP.open("rb") as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

print("\n=== KAGGLE ASSET READY ===")
print("asset folder:", ASSET_DIR)
print("asset zip:", ASSET_ZIP)
print("zip size MB:", round(ASSET_ZIP.stat().st_size / 1024**2, 2))
print("sha256:", sha256.hexdigest())
print("best epoch:", manifest["best_epoch"])
print("best validation mean ROC-AUC:", manifest["best_metric"])
print("labels:", len(label_columns))
print("files:")
for path in sorted(ASSET_DIR.iterdir()):
    print(" -", path.name, round(path.stat().st_size / 1024**2, 3), "MB")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint candidates ranked by their own validation mean ROC-AUC:
 - epoch=16 auc=0.857725882488 path=/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/best_model.pt
 - epoch=16 auc=0.857725882488 path=/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/important_epochs/best_epoch_016.pt
 - epoch=20 auc=0.856725140642 path=/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/latest_model.pt
 - epoch=14 auc=0.855615774729 path=/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/important_epochs/best_epoch_014.pt
 - epoch=13 auc=0.853027844034 path=/content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/important_epochs/best_epoch_013.pt
Selected checkpoint: /content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2/best_model.pt
Selected epoch: 16
Selected validation mean RO

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

Creating asset zip...

=== KAGGLE ASSET READY ===
asset folder: /content/ranzcr_dinov3_assets
asset zip: /content/ranzcr_dinov3_assets.zip
zip size MB: 458.07
sha256: d2ceaaae9ce85c8c2109e558115e1d0e010ff9250f15d60c1bad9211e9e01d30
best epoch: 16
best validation mean ROC-AUC: 0.8577258824877178
labels: 11
files:
 - asset_manifest.json 0.001 MB
 - best_model.pt 489.488 MB
 - config.json 0.001 MB


In [12]:
# Download the asset zip from Colab to your computer.
# If the browser blocks the automatic download, open Colab's Files panel,
# refresh it, and download /content/ranzcr_dinov3_assets.zip manually.

from pathlib import Path
from google.colab import files

ASSET_ZIP = Path("/content/ranzcr_dinov3_assets.zip")
if not ASSET_ZIP.is_file():
    raise FileNotFoundError("Run the asset packaging cell first.")

print("Downloading:", ASSET_ZIP)
print("Size MB:", round(ASSET_ZIP.stat().st_size / 1024**2, 2))
files.download(str(ASSET_ZIP))


Downloading: /content/ranzcr_dinov3_assets.zip
Size MB: 458.07


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import json
from pathlib import Path

import pandas as pd
import torch
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

ckpt_dir = Path(
    "/content/drive/MyDrive/Jiaozi/checkpoints/"
    "ranzcr_agent_retrained_v2"
)

history_path = ckpt_dir / "training_history.csv"
best_path = ckpt_dir / "best_model.pt"
latest_path = ckpt_dir / "latest_model.pt"
important_dir = ckpt_dir / "important_epochs"

print("=== CHECKPOINT DIRECTORY ===")
print("directory:", ckpt_dir)
print("exists:", ckpt_dir.is_dir())

if not ckpt_dir.is_dir():
    raise FileNotFoundError(ckpt_dir)

if not history_path.is_file():
    raise FileNotFoundError(history_path)

if not best_path.is_file():
    raise FileNotFoundError(best_path)

# 1. Find the epoch with the highest validation AUC in the full training history
history = pd.read_csv(history_path)

required_columns = {
    "epoch",
    "train_loss",
    "val_loss",
    "val_mean_auc",
}

missing_columns = required_columns.difference(
    history.columns
)

if missing_columns:
    raise ValueError(
        f"Training history is missing: "
        f"{sorted(missing_columns)}"
    )

best_index = history["val_mean_auc"].idxmax()
best_row = history.loc[best_index]

history_best_epoch = int(best_row["epoch"])
history_best_auc = float(
    best_row["val_mean_auc"]
)

print("\n=== BEST RESULT FROM TRAINING HISTORY ===")
print("completed epochs:", len(history))
print("best epoch:", history_best_epoch)
print("best validation mean ROC-AUC:", history_best_auc)
print("train loss at best epoch:", float(best_row["train_loss"]))
print("val loss at best epoch:", float(best_row["val_loss"]))

display(
    history.sort_values(
        "val_mean_auc",
        ascending=False,
    ).head(10)
)

# 2. Load best_model.pt and check its corresponding epoch and AUC
print("\n=== LOAD best_model.pt ===")
print(
    "file size MB:",
    round(best_path.stat().st_size / 1024**2, 2),
)

checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False,
)

checkpoint_epoch = int(
    checkpoint.get(
        "best_epoch",
        checkpoint.get("epoch", -1),
    )
)

validation = checkpoint.get("validation") or {}

checkpoint_auc = validation.get(
    "mean_roc_auc"
)

if checkpoint_auc is None:
    checkpoint_auc = checkpoint.get(
        "best_metric"
    )

checkpoint_auc = float(checkpoint_auc)

print("checkpoint epoch:", checkpoint_epoch)
print("checkpoint mean ROC-AUC:", checkpoint_auc)
print(
    "checkpoint validation:",
    json.dumps(
        validation,
        indent=2,
        ensure_ascii=False,
    ),
)

# 3. Check model weights and the 11-class classification head
state = checkpoint.get("model_state_dict")

if not isinstance(state, dict) or not state:
    raise RuntimeError(
        "model_state_dict is missing or empty"
    )

head_weight = state.get(
    "classifier.1.weight"
)

head_ok = (
    torch.is_tensor(head_weight)
    and tuple(head_weight.shape) == (11, 768)
)

print("\n=== MODEL STATE ===")
print("state tensor count:", len(state))
print(
    "backbone tensor count:",
    sum(
        key.startswith("backbone.")
        for key in state
    ),
)
print(
    "classification head shape:",
    tuple(head_weight.shape)
    if torch.is_tensor(head_weight)
    else None,
)

# 4. Check the representative checkpoint for the best epoch
expected_representative = (
    important_dir
    / f"best_epoch_{history_best_epoch:03d}.pt"
)

representatives = (
    sorted(important_dir.glob("*.pt"))
    if important_dir.is_dir()
    else []
)

print("\n=== REPRESENTATIVE CHECKPOINTS ===")
print(
    "saved representatives:",
    [path.name for path in representatives],
)
print(
    "expected best representative:",
    expected_representative,
)
print(
    "expected representative exists:",
    expected_representative.is_file(),
)

# 5. Combine the checks
epoch_matches = (
    checkpoint_epoch == history_best_epoch
)

auc_matches = (
    abs(checkpoint_auc - history_best_auc)
    < 1e-6
)

representative_ok = (
    expected_representative.is_file()
)

latest_exists = latest_path.is_file()

all_ok = (
    epoch_matches
    and auc_matches
    and representative_ok
    and head_ok
    and latest_exists
)

print("\n=== FINAL VERDICT ===")
print("best epoch matches history:", epoch_matches)
print("best AUC matches history:", auc_matches)
print("best representative saved:", representative_ok)
print("11-output classification head:", head_ok)
print("latest_model.pt exists:", latest_exists)
print("optimal checkpoint saved correctly:", all_ok)

if all_ok:
    print(
        "\nSUCCESS: best_model.pt corresponds to "
        "the highest validation mean ROC-AUC epoch "
        "and is ready for Kaggle asset packaging."
    )
else:
    print(
        "\nWARNING: the saved checkpoint does not "
        "fully match the optimal training-history epoch. "
        "Do not package it yet."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== CHECKPOINT DIRECTORY ===
directory: /content/drive/MyDrive/Jiaozi/checkpoints/ranzcr_agent_retrained_v2
exists: True

=== BEST RESULT FROM TRAINING HISTORY ===
completed epochs: 20
best epoch: 16
best validation mean ROC-AUC: 0.8577258824877178
train loss at best epoch: 0.4681503287272459
val loss at best epoch: 0.4616491648283872


,epoch,train_loss,val_loss,val_mean_auc,lr,steps,time_seconds
15,16,0.468150,0.461649,0.857726,0.000146,1553,176.871050
18,19,0.454548,0.465097,0.856868,0.000024,1553,174.476084
19,20,0.455130,0.464542,0.856725,0.000006,1553,174.870230
16,17,0.462997,0.464012,0.856482,0.000095,1553,175.861348
17,18,0.460316,0.463865,0.856317,0.000054,1553,177.063975
13,14,0.480389,0.467030,0.855616,0.000273,1553,175.647474
14,15,0.474394,0.465326,0.855097,0.000206,1553,176.522100
12,13,0.490688,0.470406,0.853028,0.000345,1553,176.780372
11,12,0.506955,0.480884,0.851356,0.000422,1553,174.751310
10,11,0.502352,0.477230,0.851108,0.000500,1553,173.031840



=== LOAD best_model.pt ===
file size MB: 489.49
checkpoint epoch: 16
checkpoint mean ROC-AUC: 0.8577258824877178
checkpoint validation: {
  "mean_roc_auc": 0.8577258824877178,
  "val_loss": 0.4616491648283872,
  "per_label_auc": {
    "ETT - Abnormal": 0.944895038167939,
    "ETT - Borderline": 0.9346101424361493,
    "ETT - Normal": 0.9846893382286153,
    "NGT - Abnormal": 0.9073529815694783,
    "NGT - Borderline": 0.9032883002313923,
    "NGT - Incompletely Imaged": 0.9649396764789759,
    "NGT - Normal": 0.9563931943630619,
    "CVC - Abnormal": 0.6351496438735087,
    "CVC - Borderline": 0.6149905651369305,
    "CVC - Normal": 0.5972551161803242,
    "Swan Ganz Catheter Present": 0.9914207106985208
  },
  "num_samples": 5250
}

=== MODEL STATE ===
state tensor count: 213
backbone tensor count: 211
classification head shape: (11, 768)

=== REPRESENTATIVE CHECKPOINTS ===
saved representatives: ['best_epoch_013.pt', 'best_epoch_014.pt', 'best_epoch_016.pt']
expected best representa